# try3 inference (0.87583 public, 0.8724 local)

Loads the saved models and scores new raw series. Nothing is trained here. 4,000 series take about 22 min on 2 CPU cores (2.6 GB RAM).

Each fold uses its top 100 features plus 36 noise-law prior columns. Three models (2 LightGBM, 1 XGBoost) are averaged over the 5 folds, then rank-blended.

Output: `id,break_probability`

In [ ]:
#settings, change these paths
SERIES = 'data/test_series.parquet'          #new series, same format as the competition file
MODEL = 'try3/model'                         #saved fold models + chosen columns + fold priors
REF = 'try3_run'                             #folder made by try3/build_features.py: frozen feature code + train-side reference tables
OUT = 'predict_try3'                         #scratch folder, must be empty
CSV = 'submission.csv'
WORKERS = 2

In [ ]:
#imports
import os, sys, json, glob, subprocess
import pandas as pd
import matplotlib.pyplot as plt
os.environ['CUDA_VISIBLE_DEVICES'] = ''   #cpu only

In [ ]:
#look at the new data
raw = pd.read_parquet(SERIES)
print(raw.shape, raw['id'].nunique(), 'series')
raw.head()

In [ ]:
#plot a few, red line is the boundary we score
ids = raw['id'].unique()[:3]
fig, ax = plt.subplots(len(ids), 1, figsize=(10, 6))
for a, i in zip(ax, ids):
    s = raw[raw['id'] == i].sort_values('time')
    a.plot(s['value'].values, lw=.7)
    a.axvline((s['period'] == 0).sum(), ls='--', c='r')
    a.set_title(str(i))
plt.tight_layout()

In [ ]:
#what the saved model uses
cols = [json.load(open(f'{MODEL}/columns_fold{k}.json')) for k in range(5)]
union = sorted({c for d in cols for c in d['top']})
print(len(union), 'distinct features over 5 folds, +', len(cols[0]['law']), 'law-prior columns per fold')
pd.Series([c.split('__')[0] for c in union]).value_counts().head(15)

In [ ]:
#run it:
#1 raw series -> ids + sequence cache (the model uses no base-feature columns)
#2 the other feature tables the model needs (same scripts as training, test side only, only the columns the models read)
#3 noise-law prior per fold from the saved fold priors
#4 load lgb x2 + xgb per fold, average folds, rank-blend, write id,break_probability
subprocess.run([sys.executable, 'try3/predict.py', '--series', SERIES, '--model', MODEL, '--ref', REF,
                '--out', OUT, '--csv', CSV, '--workers', str(WORKERS)], check=True)
print(json.load(open(f'{OUT}/timing.json'))['total'], 'seconds')

In [ ]:
#check the output
sub = pd.read_csv(CSV)
print(sub.shape)
print(sub.break_probability.describe())
sub.head()